In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder # <------------------ Esto es nuevo :)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import ColumnTransformer # <------------------ Esto es nuevo :)
from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline 
from sklearn.pipeline import Pipeline # <------------------ Esto es nuevo :)

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

df = pd.read_csv('./data/titanic.csv')

# Definir preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Sex', 'Embarked']),  # OneHotEncoder para columnas categóricas
        ('age', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Age']),
        ('fare', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Fare'])
    ],
    remainder='passthrough'  # Mantener otras columnas sin cambios
)

# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'model__C': [0.01, 0.1, 1, 10, 100],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear', 'saga'],
            'model__max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}


# --- 1. CARGA Y DIVISIÓN DE DATOS ---
# Usamos titanic.csv, pero debemos eliminar columnas de texto irrelevantes
# Esto evita el ValueError: could not convert string to float
columnas_no_numericas = ['Survived', 'Name', 'Ticket', 'PassengerId', 'Cabin']
X = df.drop(columnas_no_numericas, axis=1, errors='ignore')
y = df['Survived']

# Limpieza rápida de valores nulos para evitar errores en el Pipeline
X['Age'] = X['Age'].fillna(X['Age'].median())
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])
X['Fare'] = X['Fare'].fillna(X['Fare'].median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

# --- 2. ENTRENAMIENTO (CICLO FOR) ---
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None

for nombre, info_modelo in modelos.items():
    # El Pipeline integra el preprocesamiento, escalamiento y el modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),
        ('model', info_modelo['modelo'])
    ])
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    # Si X_train ya no tiene texto, esto funcionará sin errores
    grid_search.fit(X_train, y_train)
    y_pred = grid_search.predict(X_test)
    precision = accuracy_score(y_test, y_pred)
    
    # Aquí es donde se crea la columna 'Precisión' que antes faltaba
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# --- 3. RESULTADOS Y EXPORTACIÓN ---
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)
print(metricas.round(2))

# Guardar el mejor pipeline completo para usarlo en el API de Flask
with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

C:\Users\gears\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\gears\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


                                 Modelo  Precisión
6      Clasificador K-Nearest Neighbors       0.83
8                     Clasificador LGBM       0.82
3    Clasificador de Bosques Aleatorios       0.82
7                  Clasificador XGBoost       0.81
1   Clasificador de Vectores de Soporte       0.81
0                   Regresión Logística       0.81
5                 Clasificador AdaBoost       0.81
2     Clasificador de Árbol de Decisión       0.80
4     Clasificador de Gradient Boosting       0.80
9                            GaussianNB       0.79
10             Clasificador Naive Bayes       0.78
